# Clinical text embeddings

Exercise notebook for the lesson **[Clinical text embeddings](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings)** in the *Clinical NLP* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Text embeddings turn notes into vectors so that notes with similar meaning are close together, which lets a classifier recognise a problem described in words it was never trained on. On synthetic notes, an embedding-based classifier recognised foot problems in new wording far better than word counts did. But the same classifier gave 'denies tingling in both feet' the same score as 'reports tingling in both feet': embeddings capture the topic more than the assertion. Clinical use needs negation handling and validation on real local notes.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Embeddings as features

Synthetic notes; train on one set of phrases, test on paraphrases. TF-IDF baseline.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
openers = ["Seen in diabetes clinic.", "Annual review.", "Follow-up visit.", "Telephone consultation."]
foot_train = ["tingling in both feet", "numbness in the toes", "foot ulcer on the heel",
              "burning pain in the feet at night", "reduced sensation on monofilament testing"]
other_train = ["cough for one week", "headache since Monday", "medication review only",
               "blurred vision when reading", "knee pain after walking"]
# Test notes describe the same problems in words the training notes never use
foot_test = ["pins and needles in her soles", "cannot feel the floor properly",
             "sore on the sole that will not heal", "feet feel like walking on cotton wool",
             "wound under the big toe"]
other_test = ["chest infection improving", "migraine episodes", "routine prescription check",
              "eyes tired with screen use", "hip stiffness in the morning"]

def make_notes(phrases_pos, phrases_neg, n):
    notes, labels = [], []
    for _ in range(n):
        label = rng.integers(0, 2)
        phrase = rng.choice(phrases_pos if label else phrases_neg)
        notes.append(f"{rng.choice(openers)} Patient reports {phrase}. HbA1c {rng.uniform(6, 10):.1f}%.")
        labels.append(label)
    return notes, np.array(labels)

train_notes, y_train = make_notes(foot_train, other_train, 200)
test_notes, y_test = make_notes(foot_test, other_test, 200)

tfidf = TfidfVectorizer().fit(train_notes)
clf = LogisticRegression(max_iter=1000).fit(tfidf.transform(train_notes), y_train)
print(f"TF-IDF + logistic regression:     test AUROC "
      f"{roc_auc_score(y_test, clf.predict_proba(tfidf.transform(test_notes))[:, 1]):.3f}")

# TF-IDF + logistic regression:     test AUROC 0.825

### Embeddings recognise paraphrase

Continuing from the previous block: the same classifier on sentence embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E_train = encoder.encode(train_notes, normalize_embeddings=True)
E_test = encoder.encode(test_notes, normalize_embeddings=True)
clf_e = LogisticRegression(max_iter=1000).fit(E_train, y_train)
print(f"Sentence embeddings + logistic regression: test AUROC "
      f"{roc_auc_score(y_test, clf_e.predict_proba(E_test)[:, 1]):.3f}")

# Sentence embeddings + logistic regression: test AUROC 1.000

### Embeddings can ignore negation

Continuing from the previous block: three probe notes, including two that say there is no problem.

In [ ]:
probes = ["Annual review. Patient reports tingling in both feet. HbA1c 7.2%.",
          "Annual review. Patient denies tingling in both feet. HbA1c 7.2%.",
          "Annual review. No numbness, no ulcers, feet examined and normal. HbA1c 7.2%."]
for text, p in zip(probes, clf_e.predict_proba(encoder.encode(probes, normalize_embeddings=True))[:, 1]):
    print(f"P(foot problem) {p:.2f}  {text}")

# P(foot problem) 0.90  Annual review. Patient reports tingling in both feet. HbA1c 7.2%.
# P(foot problem) 0.90  Annual review. Patient denies tingling in both feet. HbA1c 7.2%.
# P(foot problem) 0.76  Annual review. No numbness, no ulcers, feet examined and normal. HbA1c 7.2%.

## Exercise

**Goal:** Build and stress-test embedding-based note classifiers.

**Data:** Synthetic notes generated in the notebook (Created for this course; no real patient data)

### Step 1

Reproduce the TF-IDF and embedding results and the negation probes.

In [ ]:
# Your code here


### Step 2

Add negated training notes ('denies…', 'no…') labelled as no problem, retrain, and re-run the probes.

In [ ]:
# Your code here


### Step 3

Apply the NegEx-style rule from the first lesson to remove negated sentences before embedding, and compare.

In [ ]:
# Your code here


### Step 4

Optional: replace the general model with a clinical encoder (for example Bio_ClinicalBERT with mean pooling) and compare on the paraphrase test.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.